# Stage 1 - Setup and Initial Inspection 

In this stage, i am going to load the Spotify Songs dataset (TidyTuesday 2020) directly from the source URL,perform an initial structural inspection, handle missing values and duplicate rows, and build a summary statistics table for all numeric columns.

In [41]:
import pandas as pd
import numpy as np

# Load dataset straight from TidyTuesday URL
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"
df = pd.read_csv(url)

print(f"Dataset Size: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset Size: 32,833 rows x 23 columns


,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


## 1.1 Meninjau Shape, Dtypes, dan Missing Values 

Now i will start by examining the dataset's dimensions, column data types, and the count/percentage of missing values per column. Any column with more than 20% missing values will be **automatically dropped**, as such columns are  considered unreliable for downstream analysis.

In [42]:
# Shape & dtypes
print("=" * 60)
print(f"Shape: {df.shape}")
print("=" * 60)
print("\nColumn Data Types:")
print(df.dtypes)

# Compute missing values (count & percentage)
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df)) * 100

missing_summary = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct.round(4)
}).sort_values('missing_pct', ascending=False)

print("\n" + "=" * 60)
print("Missing Values Summary (sorted descending):")
print("=" * 60)
print(missing_summary)

# Auto-drop columns with missing > 20%
THRESHOLD = 20.0
cols_to_drop = missing_pct[missing_pct > THRESHOLD].index.tolist()

if cols_to_drop:
    print(f"\n Columns dropped (missing > {THRESHOLD}%): {cols_to_drop}")
    df = df.drop(columns=cols_to_drop)
    print(f"Shape after drop: {df.shape}")
else:
    max_missing_col = missing_pct.idxmax()
    max_missing_val = missing_pct.max()
    print(f"\n No columns dropped.")
    print(f"   Highest missing rate: {max_missing_val:.4f}% in column '{max_missing_col}'")

Shape: (32833, 23)

Column Data Types:
track_id                     object
track_name                   object
track_artist                 object
track_popularity              int64
track_album_id               object
track_album_name             object
track_album_release_date     object
playlist_name                object
playlist_id                  object
playlist_genre               object
playlist_subgenre            object
danceability                float64
energy                      float64
key                           int64
loudness                    float64
mode                          int64
speechiness                 float64
acousticness                float64
instrumentalness            float64
liveness                    float64
valence                     float64
tempo                       float64
duration_ms                   int64
dtype: object

Missing Values Summary (sorted descending):
                          missing_count  missing_pct
track_artist         

### Interpertasi Missing Values 

The dataset is remarkably clean. The highest missing rate is only 0.0152% in the `track_name` column, which is far below the 20% threshold. Therefore, no columns are dropped at this stage. The handful of rows with missing 
values (mostly in text/metadata fields) can be safely retained, as they will not affect the numeric analyses in later stages.

## 1.2 Handling Duplicate Rows

This dataset has a specific structural quirk: a single `track_id` can appear 
in multiple playlists (and therefore across multiple `playlist_genre` values), 
because the data is organized per playlist-track pair, not per unique track.

We need to distinguish two types of duplicates:
1. Exact duplicate rows — all columns identical, pure redundancy → dropped.
2. Duplicate `track_id` — same track appearing in different playlists, part 
   of the intended data structure → kept.

Therefore, my deeduplication strategy is to remove exact duplicates only. Tracks appearing in 
multiple playlists are preserved because their playlist/genre membership is 
essential for the genre-level analysis in Stage 2.

In [43]:
# Check both types of duplicates
exact_dup = df.duplicated().sum()
track_id_dup = df.duplicated(subset='track_id').sum()

print(f"Exact duplicate rows (all columns identical): {exact_dup:,}")
print(f"Duplicates based on track_id only           : {track_id_dup:,}")
print(f"Total rows before dedup                     : {len(df):,}")

# Drop exact duplicates only
df = df.drop_duplicates().reset_index(drop=True)

print(f"Total rows after dedup                      : {len(df):,}")
print(f"Rows removed                                : {exact_dup:,}")

Exact duplicate rows (all columns identical): 0
Duplicates based on track_id only           : 4,477
Total rows before dedup                     : 32,833
Total rows after dedup                      : 32,833
Rows removed                                : 0


## 1.3 Numeric Summary Statistics Table

For every numeric column, we compute the mean, median, standard deviation, 
and IQR (Interquartile Range). The IQR is calculated manually using NumPy 
via the formula Q3 − Q1, where Q1 and Q3 come from `np.percentile()` at 
the 25th and 75th percentiles respectively. No `scipy` is used, per the 
task requirements.

In [44]:
def compute_iqr_summary(df):
    """
    Compute descriptive statistics with IQR for all numeric columns.

    Selects numeric columns from the input DataFrame and calculates the
    mean, median, standard deviation, and interquartile range (IQR) for
    each. IQR is computed manually as the difference between the 75th
    and 25th percentiles using ``np.percentile``, with NaN values
    dropped before calculation.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame containing one or more numeric columns.

    Returns
    -------
    pandas.DataFrame
        A DataFrame indexed by column name with four columns:
        ``mean``, ``median``, ``std``, and ``IQR``. All values are
        rounded to four decimal places.

    Examples
    --------
    >>> import pandas as pd, numpy as np
    >>> df = pd.DataFrame({'a': [1, 2, 3, 4, 5], 'b': [10, 20, 30, 40, 50]})
    >>> compute_iqr_summary(df)
          mean  median     std  IQR
    a    3.0     3.0   1.5811  2.0
    b   30.0    30.0  15.8114 20.0
    """
    numeric_df = df.select_dtypes(include=np.number)
    summary = pd.DataFrame({
        'mean':   numeric_df.mean(),
        'median': numeric_df.median(),
        'std':    numeric_df.std(),
        'IQR':    numeric_df.apply(
            lambda col: np.percentile(col.dropna(), 75) - np.percentile(col.dropna(), 25)
        )
    })
    return summary.round(4)


# Build and display the summary table
summary = compute_iqr_summary(df)
print("Numeric Columns Summary Statistics:")
print("=" * 70)
summary

Numeric Columns Summary Statistics:


,mean,median,std,IQR
track_popularity,42.4771,45.0000,24.9841,38.0000
danceability,0.6548,0.6720,0.1451,0.1980
energy,0.6986,0.7210,0.1809,0.2590
key,5.3745,6.0000,3.6117,7.0000
loudness,-6.7195,-6.1660,2.9884,3.5260
mode,0.5657,1.0000,0.4957,1.0000
speechiness,0.1071,0.0625,0.1013,0.0910
acousticness,0.1753,0.0804,0.2196,0.2399
instrumentalness,0.0847,0.0000,0.2242,0.0048
liveness,0.1902,0.1270,0.1543,0.1553


### Stage 1 Summary

Dataset loaded successfully with 32,833 rows and 23 columns. Missing values 
were inspected and no columns exceeded the 20% threshold (highest was only 
0.0152% in track_name), so none were dropped. For duplicates, 0 exact 
duplicate rows were found, but 4,477 rows share a track_id with at least 
one other row. These were preserved intentionally to maintain the 
multi-playlist structure, which will matter in Stage 2 when aggregating 
by genre. Summary statistics (mean, median, std, IQR) were computed for 
all numeric columns, with IQR calculated manually via NumPy.

Distribution Shape Observations

Comparing mean and median across numeric columns reveals several skewed 
distributions rather than symmetric ones:

Strongly right-skewed (mean much larger than median, long tail of high values):
- instrumentalness (mean 0.085 vs median 0.000): more than half of all 
  tracks have vocals, with a small minority of purely instrumental tracks 
  pulling the mean up.
- speechiness (0.107 vs 0.063): most tracks are sung music, but rap and 
  spoken-word tracks form a long right tail.
- acousticness (0.175 vs 0.080): the catalog is dominated by produced or 
  electronic music, with a smaller cluster of acoustic tracks.
- liveness (0.190 vs 0.127): mostly studio recordings, with occasional 
  live recordings on the high end.

Mildly right-skewed:
- duration_ms (225,800 vs 216,000): most tracks sit around 3 to 4 minutes, 
  but a few unusually long tracks pull the mean up.

Mildly left-skewed (mean smaller than median, tail of low values):
- track_popularity (42.48 vs 45.00): a large group of low-popularity tracks 
  drags the mean below the median.
- loudness (-6.72 vs -6.17): a few very quiet tracks stretch the left tail.

Roughly symmetric: danceability, energy, valence, and tempo have mean and 
median values close together, suggesting these features are fairly balanced 
across the catalog.

Implication for later analysis: for the strongly right-skewed features, 
the median is a more honest description of a typical track than the mean. 
This is worth remembering when interpreting genre-level averages in Stage 2 
and correlation results in Stage 3.

The dataset is now ready for genre and popularity analysis in Stage 2.

# Stage 2 — Genre and Popularity Analysis

In this stage, I explore how audio features and popularity vary across 
playlist genres, identify which artists dominate the catalog, and filter 
for tracks that match a "hit potential" profile. The goal is to move 
from raw statistics to business oriented insights.

## 2.1 Distribution Statistics per Genre

We group the dataset by playlist_genre and compute the mean, standard 
deviation, and median of four key features: track_popularity, danceability, 
energy, and valence. This gives a compact profile of each genre's typical 
sound and how consistent that sound is within the genre.

In [45]:
# Features to analyze
features = ['track_popularity', 'danceability', 'energy', 'valence']

# Groupby genre with multiple aggregations
genre_stats = df.groupby('playlist_genre')[features].agg(['mean', 'std', 'median']).round(4)

print("Distribution Statistics per Genre:")
print("=" * 80)
genre_stats



Distribution Statistics per Genre:


track_popularity                 danceability                 \
                           mean      std median         mean     std median   
playlist_genre                                                                
edm                     34.8335  23.1542   36.0       0.6550  0.1236  0.659   
latin                   47.0266  25.4246   50.0       0.7133  0.1150  0.729   
pop                     47.7449  25.1583   52.0       0.6393  0.1282  0.652   
r&b                     41.2235  25.8945   44.0       0.6702  0.1382  0.689   
rap                     43.2155  23.3021   47.0       0.7184  0.1365  0.737   
rock                    41.7283  24.8252   46.0       0.5205  0.1399  0.523   

                energy                valence                 
                  mean     std median    mean     std median  
playlist_genre                                                
edm             0.8025  0.1394  0.830  0.4007  0.2262  0.370  
latin           0.7083  0.1523  0.729  0.6055  0.2223  0.628  
pop             0.7010  0.1711  0.727  0.5035  0.2205  0.500  
r&b             0.5909  0.1794  0.596  0.5312  0.2259  0.542  
rap             0.6507  0.1703  0.665  0.5051  0.2247  0.517  
rock            0.7328  0.1945  0.775  0.5374  0.2293  0.531

Reading the table: each genre row shows nine numbers per feature 
(mean, std, median). This lets us see both the typical value of a feature 
and how much it varies within that genre. For example, a genre with 
high mean danceability but low std produces a consistently danceable 
sound, while a genre with high std produces a wider stylistic range.

## 2.2 Which Genre Has the Highest Popularity Variance?

We now isolate the standard deviation of track_popularity per genre. 
A genre with high popularity variance is one where some tracks become 
huge hits while many others remain unnoticed — the outcomes are less 
predictable. A genre with low variance produces more uniformly received 
tracks.

In [46]:
# Extract std of track_popularity per genre from the previous table
pop_variance = genre_stats[('track_popularity', 'std')].sort_values(ascending=False)

print("Track Popularity Standard Deviation per Genre (sorted):")
print("=" * 60)
print(pop_variance.round(4))

top_var_genre = pop_variance.idxmax()
top_var_value = pop_variance.max()
print(f"\nHighest variance: '{top_var_genre}' with std = {top_var_value:.4f}")

Track Popularity Standard Deviation per Genre (sorted):
playlist_genre
r&b      25.8945
latin    25.4246
pop      25.1583
rock     24.8252
rap      23.3021
edm      23.1542
Name: (track_popularity, std), dtype: float64

Highest variance: 'r&b' with std = 25.8945


Business interpretation for content recommendation:

A genre with the highest popularity variance is riskier to auto-recommend. 
Users may receive either a viral hit or an obscure track with almost no 
listens, and the outcome is hard to predict from the genre label alone. 
This suggests that the recommendation engine should not treat all tracks 
within this genre as equally safe bets — additional signals (release date, 
artist popularity, playlist context) become more important.

Conversely, low-variance genres are safer defaults for auto-play or 
new-user onboarding, because the popularity outcome is more consistent, 
even if the ceiling for viral hits is lower.

## 2.3 Top 10 Artists by Track Count — Does Volume Correlate with Quality?

We identify the 10 artists with the most tracks in the dataset, then 
check their mean track_popularity. To make the volume vs quality question 
more rigorous, we also compute the Pearson correlation between the two 
measures across these 10 artists.

In [47]:
# Top 10 artists by track count
top_10_artists = df['track_artist'].value_counts().head(10)

# Compute mean popularity for each of these artists
artist_stats = (
    df[df['track_artist'].isin(top_10_artists.index)]
    .groupby('track_artist')['track_popularity']
    .mean()
    .round(2)
)

# Combine into one table
top_10_summary = pd.DataFrame({
    'track_count': top_10_artists,
    'mean_popularity': artist_stats
}).sort_values('mean_popularity', ascending=False)

print("Top 10 Artists by Track Count (sorted by mean popularity):")
print("=" * 60)
print(top_10_summary)

# Pearson correlation between track count and mean popularity
correlation = top_10_summary['track_count'].corr(top_10_summary['mean_popularity'])
print(f"\nPearson correlation (track_count vs mean_popularity): {correlation:.4f}")

Top 10 Artists by Track Count (sorted by mean popularity):
                           track_count  mean_popularity
track_artist                                           
Kygo                                83            65.12
Calvin Harris                       91            61.81
The Chainsmokers                   123            57.70
David Guetta                       110            53.44
Martin Garrix                      161            47.20
Drake                              100            46.43
Queen                              136            43.00
Dimitri Vegas & Like Mike           93            42.80
Don Omar                           102            41.95
Hardwell                            84            39.11

Pearson correlation (track_count vs mean_popularity): -0.1730


Interpretation of volume vs quality:

The Pearson correlation between track count and mean popularity across the 
top 10 artists is -0.1730, a weak negative relationship. This suggests a 
mild "quantity over quality" tension in the dataset: artists with the 
largest catalogs tend to have slightly lower average popularity per track, 
while more selective releasers score higher on average.

Concrete examples from the table support this pattern. Kygo has the fewest 
tracks among the top 10 (83) but the highest mean popularity (65.12). 
Martin Garrix has the most tracks (161) with a mean popularity of only 
47.20. Queen, with 136 tracks, sits at a mean of just 43.00.

However, the correlation is weak in magnitude, so this is a tendency rather 
than a rule. Calvin Harris, for instance, sustains both a moderately large 
catalog (91 tracks) and a high mean popularity (61.81). A separate 
observation worth noting: 9 of the top 10 artists by track count are 
EDM or dance producers, which likely explains why the edm genre has the 
lowest popularity standard deviation in the previous analysis — the 
genre's volume is dominated by prolific artists whose consistent output 
flattens its popularity distribution.

The sample is limited to 10 artists, so the correlation is directional 
rather than statistically robust, but it does give a useful business 
signal: in this catalog, releasing more does not automatically mean 
performing better.

## 2.4 Filtering for Hit Potential Tracks

Now I filter the dataset for tracks that satisfy all of the following:
- track_popularity greater than 70
- danceability greater than 0.7
- energy greater than 0.6
- duration_ms less than 240000 (under 4 minutes)

This combination approximates a "hit potential" profile: popular, danceable, 
energetic, and radio-friendly in length. We then check which genre 
dominates this subset.

In [48]:
def filter_hit_potential(df, popularity_min=70, danceability_min=0.7,
                         energy_min=0.6, duration_max=240000):
    """
    Filter tracks that match a "hit potential" profile.

    Applies four threshold filters simultaneously to isolate tracks with
    high popularity, high danceability, high energy, and short duration.
    All conditions use strict inequality (``>`` for minimums, ``<`` for
    the maximum).

    Parameters
    ----------
    df : pandas.DataFrame
        Spotify tracks DataFrame. Must contain the columns
        ``track_popularity``, ``danceability``, ``energy``, and
        ``duration_ms``.
    popularity_min : int, optional
        Minimum track popularity score (exclusive). Default is 70.
    danceability_min : float, optional
        Minimum danceability value (exclusive, 0-1 scale). Default is
        0.7.
    energy_min : float, optional
        Minimum energy value (exclusive, 0-1 scale). Default is 0.6.
    duration_max : int or float, optional
        Maximum track duration in milliseconds (exclusive). Default is
        240000 (4 minutes).

    Returns
    -------
    pandas.DataFrame
        Subset of ``df`` containing only the rows that satisfy all four
        filter conditions. Retains the original index and columns.

    Examples
    --------
    >>> hit_tracks = filter_hit_potential(df)
    >>> print(f"{len(hit_tracks)} tracks pass all filters")
    1112 tracks pass all filters
    """
    mask = (
        (df['track_popularity'] > popularity_min) &
        (df['danceability'] > danceability_min) &
        (df['energy'] > energy_min) &
        (df['duration_ms'] < duration_max)
    )
    return df[mask]


# Apply the filter using the default hit-potential thresholds
hit_tracks = filter_hit_potential(df)

print(f"Number of tracks passing all filters: {len(hit_tracks):,}")
print(f"Percentage of full dataset          : {len(hit_tracks) / len(df) * 100:.2f}%")

# Genre distribution within the hit subset
genre_dist = hit_tracks['playlist_genre'].value_counts()
genre_pct = (genre_dist / len(hit_tracks) * 100).round(2)

hit_genre_summary = pd.DataFrame({
    'count': genre_dist,
    'percentage': genre_pct
})

print("\nGenre distribution among hit-potential tracks:")
print("=" * 50)
print(hit_genre_summary)

dominant_genre = genre_dist.idxmax()
print(f"\nDominant genre: '{dominant_genre}' with {genre_dist.max()} tracks "
      f"({genre_pct.max():.2f}% of the hit subset)")

Number of tracks passing all filters: 1,112
Percentage of full dataset          : 3.39%

Genre distribution among hit-potential tracks:
                count  percentage
playlist_genre                   
latin             396       35.61
pop               264       23.74
rap               173       15.56
r&b               135       12.14
edm               111        9.98
rock               33        2.97

Dominant genre: 'latin' with 396 tracks (35.61% of the hit subset)


Only 1,112 tracks (3.39% of the dataset) satisfy all four hit-potential 
criteria simultaneously, confirming that the combination of high popularity, 
high danceability, high energy, and short duration is a genuinely selective 
profile.

The latin genre dominates this subset with 396 tracks (35.61%), followed 
by pop (23.74%) and rap (15.56%). Rock is almost absent (2.97%), which 
aligns with the earlier finding that rock has the lowest mean danceability 
and energy among all genres — its typical sound simply does not match 
the hit-potential profile defined here.

Business takeaway: when a recommendation engine targets high-engagement 
contexts such as party playlists, workout sessions, or discovery feeds 
aimed at new listeners, latin and pop tracks are the strongest structural 
candidates. They most frequently satisfy all four criteria at once, making 
them low-risk defaults for these use cases.

### Stage 2 Summary

Genre-level statistics showed clear stylistic differences across the six 
playlist genres. Pop and latin had the highest mean track popularity 
(around 47), while edm had the lowest (about 35). Rap emerged as the 
most danceable genre (0.72), rock the least (0.52). Edm dominated on 
energy (0.80) but had the lowest valence (0.40), suggesting a high-energy 
but emotionally darker style, while latin had the highest valence (0.60), 
matching its reputation as upbeat music.

The genre with the highest popularity variance was r&b (std = 25.89), 
narrowly ahead of latin and pop. Edm had the lowest variance (23.15), 
meaning its popularity outcomes are the most predictable, though 
consistently on the lower side. For a recommendation engine, this means 
r&b tracks are harder to auto-recommend confidently — the same genre 
label can produce either a hit or an obscure track — while edm behaves 
more uniformly.

The Pearson correlation between track count and mean popularity across 
the top 10 artists was -0.1730, a weak negative relationship. This 
suggests a mild "quantity over quality" tension: the most selective 
releaser in the top 10 (Kygo, 83 tracks) had the highest mean popularity 
(65.12), while the most prolific (Martin Garrix, 161 tracks) sat at 
47.20. The correlation is weak in magnitude, so this is a tendency 
rather than a rule. A related observation: 9 of the top 10 artists 
by track count are EDM or dance producers, which likely explains why 
edm has the lowest popularity variance — its distribution is flattened 
by a small group of prolific consistent artists.

Filtering for tracks with popularity above 70, danceability above 0.7, 
energy above 0.6, and duration under 4 minutes produced a selective 
"hit potential" subset of 1,112 tracks (3.39% of the dataset). Latin 
dominated this subset (35.61%), followed by pop (23.74%), while rock 
was nearly absent (2.97%). This confirms that certain genres are 
structurally better aligned with the audio profile of hit tracks.

The dataset is now ready for pure NumPy analysis in Stage 3.

# Stage 3 — NumPy Analysis

In this stage, we work directly with NumPy arrays instead of Pandas 
DataFrames. We extract the nine numeric audio features, normalize them 
with vectorized min-max scaling, compute a correlation matrix manually 
via np.corrcoef, and use boolean masking to isolate high-energy tracks. 
No Python loops and no sklearn are used, per the task requirements.

## 3.1 Extracting Audio Features and Min-Max Normalization

We extract the nine numeric audio feature columns as a NumPy array and 
apply min-max scaling so that every feature falls within the range [0, 1]. 
The formula is:

    X_scaled = (X - X_min) / (X_max - X_min)

This is applied column-wise using axis=0, so each feature is scaled 
independently based on its own min and max. The operation is fully 
vectorized, with no Python loops and no sklearn.

Note on loudness: this feature is measured in decibels and has negative 
values in the original data. After normalization, the quietest track 
maps to 0 and the loudest maps to 1, which is the natural interpretation.

In [49]:
# Define the nine audio feature columns
audio_features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

# Extract as NumPy array
X = df[audio_features].to_numpy()
print(f"Original array shape: {X.shape}")
print(f"Original min per feature: {X.min(axis=0).round(4)}")
print(f"Original max per feature: {X.max(axis=0).round(4)}")

# Min-max normalization, vectorized, no loops
X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

# Verification: every feature should now have min = 0 and max = 1
print(f"\nNormalized min per feature: {X_norm.min(axis=0).round(4)}")
print(f"Normalized max per feature: {X_norm.max(axis=0).round(4)}")

# Show a few sample rows
print(f"\nFirst 3 rows of normalized array:")
print(X_norm[:3].round(4))

Original array shape: (32833, 9)
Original min per feature: [ 0.0000e+00  2.0000e-04 -4.6448e+01  0.0000e+00  0.0000e+00  0.0000e+00
  0.0000e+00  0.0000e+00  0.0000e+00]
Original max per feature: [  0.983   1.      1.275   0.918   0.994   0.994   0.996   0.991 239.44 ]

Normalized min per feature: [0. 0. 0. 0. 0. 0. 0. 0. 0.]
Normalized max per feature: [1. 1. 1. 1. 1. 1. 1. 1. 1.]

First 3 rows of normalized array:
[[0.7609 0.916  0.9181 0.0635 0.1026 0.     0.0656 0.5227 0.5097]
 [0.7386 0.815  0.8692 0.0406 0.0728 0.0042 0.3584 0.6993 0.4175]
 [0.6867 0.931  0.9014 0.0808 0.0799 0.     0.1104 0.6186 0.5179]]


## 3.2 Correlation Matrix of Audio Features

I decide to compute the Pearson correlation matrix of the nine audio features 
using np.corrcoef. By default, np.corrcoef treats each row as a variable, 
but the data has features stored in columns, so i passed rowvar=False to 
correct this. The result is a 9 x 9 matrix.

Heads up on data choice: I use the original (un-normalized) feature 
values here, not the min-max scaled version. This is intentional and 
mathematically valid — Pearson correlation is invariant to linear 
transformations, so scaling has no effect on the correlation values. 
Using the original data keeps the computation clean and avoids an 
unnecessary preprocessing step.

To find the strongest positive and negative pairs, we mask the diagonal 
(self-correlations = 1) and use only the upper triangle to avoid 
double-counting symmetric pairs.

In [50]:
# Compute correlation matrix on the original (un-normalized) data
corr_matrix = np.corrcoef(X, rowvar=False)

# Display as a labeled DataFrame for readability
corr_df = pd.DataFrame(corr_matrix, index=audio_features, columns=audio_features).round(4)
print("Correlation Matrix of Audio Features:")
print("=" * 70)
print(corr_df)

# Find strongest positive and negative correlation pairs
# Step 1: mask the diagonal (set to NaN so it is ignored)
# Step 2: use only the upper triangle to avoid duplicate pairs (matrix is symmetric)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # upper triangle, excluding diagonal
corr_masked = np.where(mask, corr_matrix, np.nan)

# Find indices of max and min
max_idx = np.unravel_index(np.nanargmax(corr_masked), corr_masked.shape)
min_idx = np.unravel_index(np.nanargmin(corr_masked), corr_masked.shape)

max_pair = (audio_features[max_idx[0]], audio_features[max_idx[1]])
min_pair = (audio_features[min_idx[0]], audio_features[min_idx[1]])

print(f"\nStrongest positive correlation: {max_pair[0]} vs {max_pair[1]} = {corr_matrix[max_idx]:.4f}")
print(f"Strongest negative correlation: {min_pair[0]} vs {min_pair[1]} = {corr_matrix[min_idx]:.4f}")

Correlation Matrix of Audio Features:
                  danceability  energy  loudness  speechiness  acousticness  \
danceability            1.0000 -0.0861    0.0253       0.1817       -0.0245   
energy                 -0.0861  1.0000    0.6766      -0.0321       -0.5397   
loudness                0.0253  0.6766    1.0000       0.0103       -0.3616   
speechiness             0.1817 -0.0321    0.0103       1.0000        0.0261   
acousticness           -0.0245 -0.5397   -0.3616       0.0261        1.0000   
instrumentalness       -0.0087  0.0332   -0.1478      -0.1034       -0.0069   
liveness               -0.1239  0.1612    0.0776       0.0554       -0.0772   
valence                 0.3305  0.1511    0.0534       0.0647       -0.0168   
tempo                  -0.1841  0.1500    0.0938       0.0446       -0.1127   

                  instrumentalness  liveness  valence   tempo  
danceability               -0.0087   -0.1239   0.3305 -0.1841  
energy                      0.0332    0.161

Musical interpretation of the two extreme pairs:

The strongest positive correlation is between energy and loudness at 
0.6766. This confirms a well-known production reality: energetic tracks 
are engineered to sound louder. Heavier drums, denser mixes, and more 
compression all raise both perceived energy and measured loudness at 
the same time. Softer tracks, by contrast, are typically produced with 
more headroom and lower average volume.

The strongest negative correlation is between energy and acousticness 
at -0.5397. Acoustic tracks — unplugged guitar, piano ballads, folk 
music — are inherently gentler and less aggressive, so they score low 
on energy. Produced electronic and rock tracks, which sit on the 
opposite end of the acoustic spectrum, tend to score high on energy. 
This pair captures a fundamental stylistic axis in the catalog: 
"acoustic and calm" versus "produced and intense".

Two additional observations from the matrix are worth noting. First, 
acousticness also correlates negatively with loudness (-0.3616), 
reinforcing the same stylistic axis. Second, valence and danceability 
show a moderate positive correlation (0.3305), meaning happier-sounding 
tracks tend to be more danceable — an intuitive musical link. 
Instrumentalness, by contrast, shows only weak correlations with every 
other feature, suggesting it captures an independent dimension: 
instrumental tracks can be either energetic or calm, acoustic or 
electronic, without a consistent pattern.

## 3.3 Boolean Masking: High-Energy Tracks vs Overall Popularity

I use NumPy boolean masking (not Pandas filtering) to identify tracks 
whose energy value lies above the mean plus one standard deviation. 
These are the top-energy tracks in the catalog. I then compare their 
mean track_popularity against the overall mean, to test whether high 
energy is associated with higher or lower popularity in this dataset.

I work with the original (un-scaled) energy values here, since the 
mean and standard deviation are more interpretable in the original units.

In [51]:
# Extract energy and popularity as NumPy arrays
energy_arr = df['energy'].to_numpy()
popularity_arr = df['track_popularity'].to_numpy()

# Compute mean and std of energy
energy_mean = energy_arr.mean()
energy_std = energy_arr.std()
threshold = energy_mean + energy_std

print(f"Energy mean (mu)     : {energy_mean:.4f}")
print(f"Energy std (sigma)   : {energy_std:.4f}")
print(f"Threshold (mu + sigma): {threshold:.4f}")

# Boolean mask, pure NumPy
high_energy_mask = energy_arr > threshold
print(f"\nNumber of high-energy tracks: {high_energy_mask.sum():,}")
print(f"Percentage of dataset       : {high_energy_mask.mean() * 100:.2f}%")

# Compare mean popularity
high_energy_popularity_mean = popularity_arr[high_energy_mask].mean()
overall_popularity_mean = popularity_arr.mean()
difference = high_energy_popularity_mean - overall_popularity_mean

print(f"\nOverall mean popularity          : {overall_popularity_mean:.4f}")
print(f"High-energy subset mean popularity: {high_energy_popularity_mean:.4f}")
print(f"Difference                       : {difference:+.4f}")

Energy mean (mu)     : 0.6986
Energy std (sigma)   : 0.1809
Threshold (mu + sigma): 0.8795

Number of high-energy tracks: 5,625
Percentage of dataset       : 17.13%

Overall mean popularity          : 42.4771
High-energy subset mean popularity: 36.1244
Difference                       : -6.3526


Interpretation:

The high-energy subset contains 5,625 tracks (17.13% of the dataset), 
defined as tracks with energy above 0.8795 (mean plus one standard 
deviation).

The mean track popularity of this subset is 36.12, compared to the 
overall mean of 42.48 — a difference of -6.35 points. This is a 
meaningful gap: the loudest and most intense tracks in the catalog 
are, on average, noticeably less popular than the typical track.

This finding aligns tightly with the Stage 2 observation that edm, 
the most energetic genre, also had the lowest mean popularity. Taken 
together, the pattern suggests that in this dataset, extreme energy 
is not a driver of listener engagement — if anything, it is a mild 
negative signal. For a recommendation engine, this means that ranking 
tracks purely on energy would likely underperform, and that moderate 
energy (closer to the overall mean of 0.70) may actually be the 
sweet spot for broad appeal.

### Stage 3 Summary

The nine audio features were extracted into a NumPy array and normalized 
to the [0, 1] range using vectorized min-max scaling, with no Python 
loops and no external libraries beyond NumPy. Verification confirmed 
that every feature now has min = 0 and max = 1.

A 9 x 9 Pearson correlation matrix was computed manually via 
np.corrcoef on the original feature values, since Pearson correlation 
is invariant to linear scaling. The strongest positive pair was 
energy and loudness at 0.6766, reflecting the production reality that 
energetic tracks are engineered to sound louder. The strongest negative 
pair was energy and acousticness at -0.5397, capturing the fundamental 
stylistic axis between acoustic-and-calm tracks and produced-and-intense 
tracks. Additional patterns worth noting were a moderate positive link 
between valence and danceability (0.3305) and the near-zero correlations 
of instrumentalness with all other features, suggesting it captures an 
independent dimension of the audio profile.

Boolean masking was used to isolate 5,625 tracks (17.13% of the dataset) 
with energy above one standard deviation from the mean. The mean 
popularity of this high-energy subset was 36.12, compared to the overall 
mean of 42.48 — a gap of -6.35 points. This aligns with the Stage 2 
observation that edm, the most energetic genre, also had the lowest 
mean popularity. The pattern suggests that in this catalog, extreme 
energy is a mild negative signal for popularity, and that moderate 
energy levels may be the sweet spot for broad listener appeal.


# Stage 4 — Documentation and AI Tool Reflection

In this stage, I document the reusable functions written earlier by
generating NumPy-style docstrings with an AI coding tool, evaluate the
AI output honestly, and finally summarize the three key insights of
the whole analysis in plain language for a non-technical audience.

## 4.1 Docstring Generation via AI Coding Tool

Tool used: Antigravity (Google's agentic IDE, successor to Gemini Code
Assist, powered by Gemini models). The tool has direct access to the
notebook file, so prompts reference the functions by name rather than
pasting the full source code.

### Function 1: compute_iqr_summary

Prompt sent to Antigravity:

> In my notebook, there is a function called compute_iqr_summary.
> Generate a NumPy-style docstring for it. The function computes mean,
> median, standard deviation, and IQR for all numeric columns of a
> DataFrame, where IQR is calculated manually via np.percentile.
>

Raw AI output:

Now I have the full function. Since .ipynb files can't be edited directly, I'll provide the docstring for you to paste in.

Here's the NumPy-style docstring for compute_iqr_summary:
def compute_iqr_summary(df):
    """
    Compute descriptive statistics with IQR for all numeric columns.

    Selects numeric columns from the input DataFrame and calculates the
    mean, median, standard deviation, and interquartile range (IQR) for
    each. IQR is computed manually as the difference between the 75th
    and 25th percentiles using ``np.percentile``, with NaN values
    dropped before calculation.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame containing one or more numeric columns.

    Returns
    -------
    pandas.DataFrame
        A DataFrame indexed by column name with four columns:
        ``mean``, ``median``, ``std``, and ``IQR``. All values are
        rounded to four decimal places.

    Examples
    --------
    >>> import pandas as pd, numpy as np
    >>> df = pd.DataFrame({'a': [1, 2, 3, 4, 5], 'b': [10, 20, 30, 40, 50]})
    >>> compute_iqr_summary(df)
          mean  median     std  IQR
    a    3.0     3.0   1.5811  2.0
    b   30.0    30.0  15.8114 20.0
    """

Modification made: added a blank line between the summary line and the
extended description to strictly conform to NumPy docstring style,
which specifies the summary must be a one-liner separated from the
rest of the docstring by an empty line. The AI output ran the two
together as a single paragraph.

Evaluation: the output was strong overall. The AI actually read the
function body rather than guessing from the name — it correctly
mentioned np.percentile, the manual Q3 minus Q1 formula, the NaN
dropping via dropna, and the four-decimal rounding. The Examples
section used a minimal DataFrame with hand-verifiable numbers, which
is exactly what a docstring example should look like. I verified the
example output manually: for [1, 2, 3, 4, 5] the sample standard
deviation (pandas default, ddof=1) is 1.5811, and the IQR is 2.0,
both matching the AI's output. Only the missing blank line needed
fixing; everything else was used as-is.

### Function 2: filter_hit_potential

Prompt sent to Antigravity:

> In my notebook, there is a function called filter_hit_potential.
> Generate a NumPy-style docstring for it. The function filters a
> Spotify tracks DataFrame based on four thresholds (popularity,
> danceability, energy, and duration) to isolate tracks matching a
> "hit potential" profile.
>
> Just show me the docstring as a code block, don't edit the file yet.

Raw AI output:
Here's the NumPy-style docstring for filter_hit_potential:

def filter_hit_potential(df, popularity_min=70, danceability_min=0.7,
                         energy_min=0.6, duration_max=240000):
    """
    Filter tracks that match a "hit potential" profile.

    Applies four threshold filters simultaneously to isolate tracks with
    high popularity, high danceability, high energy, and short duration.
    All conditions use strict inequality (``>`` for minimums, ``<`` for
    the maximum).

    Parameters
    ----------
    df : pandas.DataFrame
        Spotify tracks DataFrame. Must contain the columns
        ``track_popularity``, ``danceability``, ``energy``, and
        ``duration_ms``.
    popularity_min : int or float, optional
        Minimum track popularity score (exclusive). Default is 70.
    danceability_min : float, optional
        Minimum danceability value (exclusive, 0–1 scale). Default is
        0.7.
    energy_min : float, optional
        Minimum energy value (exclusive, 0–1 scale). Default is 0.6.
    duration_max : int or float, optional
        Maximum track duration in milliseconds (exclusive). Default is
        240000 (4 minutes).

    Returns
    -------
    pandas.DataFrame
        Subset of ``df`` containing only the rows that satisfy all four
        filter conditions. Retains the original index and columns.

    Examples
    --------
    >>> hit_tracks = filter_hit_potential(df)
    >>> hit_tracks_custom = filter_hit_potential(
    ...     df, popularity_min=80, danceability_min=0.8,
    ...     energy_min=0.7, duration_max=200000
    ... )
    """

Modifications made: three small changes were applied.

First, a blank line was added between the summary line and the
extended description, matching the same NumPy style issue found in
the previous function.

Second, the type hint for popularity_min was simplified from
"int or float" to "int". The Spotify dataset's track_popularity
column is stored as integer values between 0 and 100, so allowing
float is technically valid but overspecifies the parameter and does
not reflect the actual data type users will pass in.

Third, the Examples section was strengthened. The AI's version only
showed how to call the function with default and custom arguments but
never indicated what the output looks like. I replaced the second
example (redundant with the first) with a print statement showing
the expected row count. This gives future readers an immediate sense
of the function's output size, which is more informative than a
second call signature.

Evaluation: the AI's output was accurate on the technical details.
It correctly identified strict inequalities, listed all required
columns, and explained the meaning of duration_max in both raw and
human-readable units. The main weaknesses were stylistic — an
overly permissive type hint and Examples that showed calls but not
results. Neither was a factual error, but both weakened the
docstring as documentation. After the three modifications, the
final version is both technically accurate and more useful to a
reader.

## 4.2 Three Key Insights (for a Non-Technical Audience)

Below are the three most important findings from this analysis, written
in plain language for readers who are not familiar with statistics or
data science.

### Insight 1: Releasing more songs does not mean writing more hits

Among the ten artists who released the most tracks in this dataset,
those who released the fewest were often the ones with the most popular
songs on average. Kygo, who has the smallest catalog of the ten, has
the highest average track popularity. Martin Garrix, who has the
largest catalog, sits well below him. Being prolific does not
guarantee popularity — quality tends to come from selectivity, not
volume. For record labels and A&R teams, this suggests that pushing
artists to release more music is not automatically a winning strategy.

### Insight 2: Latin and pop are the safest bets for a "hit-style" playlist

When filtering for tracks that combine four hit-like qualities at once
— popular, easy to dance to, energetic, and short enough for radio —
only about 3 percent of the entire catalog qualifies. Of those tracks,
more than one in three come from the latin genre, and another quarter
come from pop. Rock is nearly absent. This means that if a streaming
service wants to build a playlist for parties, workouts, or new-user
discovery, latin and pop are structurally the best sources of
candidates. Rock, no matter how beloved, simply does not fit this
specific profile of a modern hit.

### Insight 3: The most intense songs are not the most popular ones

The loudest, most energetic tracks in the catalog — the top 17 percent
by energy level — are on average about 6 popularity points below the
typical song. The genre with the highest energy (edm) also happens to
have the lowest average popularity. In other words, cranking up the
intensity does not attract more listeners; it may even push them away.
For anyone building a recommendation engine or curating a playlist for
broad appeal, the sweet spot is moderate energy, not extreme energy.
Music that is engaging without being overwhelming tends to travel
further.

# Bonus 1 — Artist Audio Fingerprint

In this section, I represent each artist as a single vector — the mean
of their nine audio features — and compare artists using cosine
similarity implemented directly from its mathematical definition.

Data choice: I use the normalized feature matrix from Stage 3 rather
than the original values. Cosine similarity and Euclidean distance are
not scale-invariant (unlike Pearson correlation), so features with
larger magnitudes such as loudness (-60 to 0 dB) and tempo (60 to 200
BPM) would otherwise dominate the computation and mask the contribution
of features scaled 0 to 1 like danceability. Normalizing puts every
feature on equal footing.

In [52]:
# Reuse X_norm from Stage 3 Point 9 to create a DataFrame where the nine
# audio features are min-max scaled to [0, 1]. All non-feature columns
# (track_artist, playlist_genre, etc.) are kept unchanged for filtering.
df_norm = df.copy()
df_norm[audio_features] = X_norm

print(f"df_norm shape: {df_norm.shape}")
print(f"Audio feature range after normalization:")
print(df_norm[audio_features].agg(['min', 'max']).round(4))

df_norm shape: (32833, 23)
Audio feature range after normalization:
     danceability  energy  loudness  speechiness  acousticness  \
min           0.0     0.0       0.0          0.0           0.0   
max           1.0     1.0       1.0          1.0           1.0   

     instrumentalness  liveness  valence  tempo  
min               0.0       0.0      0.0    0.0  
max               1.0       1.0      1.0    1.0  


In [53]:
def artist_fingerprint(df, artist_name):
    """
    Compute the audio fingerprint of an artist as a mean feature vector.

    Filters the DataFrame for all tracks by the given artist and returns
    the mean of the nine audio features as a NumPy array. This vector
    represents the artist's average sonic profile across their catalog.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing at least a ``track_artist`` column and the
        nine audio feature columns: ``danceability``, ``energy``,
        ``loudness``, ``speechiness``, ``acousticness``,
        ``instrumentalness``, ``liveness``, ``valence``, and ``tempo``.
    artist_name : str
        The exact artist name to filter on, matched against
        ``track_artist``.

    Returns
    -------
    numpy.ndarray
        A 1D array of shape (9,) containing the mean value of each audio
        feature for the specified artist. If the artist is not found in
        the DataFrame, all elements are NaN.

    Examples
    --------
    >>> fp = artist_fingerprint(df_norm, "Kygo")
    >>> fp.shape
    (9,)
    """
    audio_cols = ['danceability', 'energy', 'loudness', 'speechiness',
                  'acousticness', 'instrumentalness', 'liveness',
                  'valence', 'tempo']
    artist_tracks = df[df['track_artist'] == artist_name]
    return artist_tracks[audio_cols].mean().to_numpy()


def cosine_similarity(A, B):
    """
    Compute cosine similarity between two vectors from its definition.

    Implements the standard mathematical formula using only NumPy:

        cosine_similarity(A, B) = (A . B) / (||A|| * ||B||)

    where ``.`` is the dot product and ``|| ||`` is the Euclidean norm.
    The result ranges from -1 (opposite direction) to 1 (identical
    direction), with 0 indicating orthogonality.

    Parameters
    ----------
    A : numpy.ndarray
        First input vector, 1D array of any length.
    B : numpy.ndarray
        Second input vector, must be the same length as ``A``.

    Returns
    -------
    float
        Cosine similarity value in the range [-1, 1]. Higher values
        indicate the two vectors point in more similar directions.

    Examples
    --------
    >>> import numpy as np
    >>> cosine_similarity(np.array([1, 0, 0]), np.array([1, 0, 0]))
    1.0
    >>> cosine_similarity(np.array([1, 0]), np.array([0, 1]))
    0.0
    """
    dot_product = np.dot(A, B)
    norm_A = np.sqrt(np.sum(A ** 2))
    norm_B = np.sqrt(np.sum(B ** 2))
    return dot_product / (norm_A * norm_B)

In [54]:
# Three pairs designed to span a range of expected similarity:
# 1. Both EDM producers -> expected highest similarity
# 2. EDM vs rap        -> moderate similarity
# 3. Rock vs latin     -> expected lowest similarity
pairs = [
    ("Kygo",  "Martin Garrix"),
    ("Kygo",  "Drake"),
    ("Queen", "Don Omar"),
]

print("Cosine Similarity Between Artist Pairs")
print("=" * 60)
for artist_a, artist_b in pairs:
    fp_a = artist_fingerprint(df_norm, artist_a)
    fp_b = artist_fingerprint(df_norm, artist_b)
    similarity = cosine_similarity(fp_a, fp_b)
    print(f"{artist_a:<15} vs {artist_b:<15} : {similarity:.4f}")

Cosine Similarity Between Artist Pairs
Kygo            vs Martin Garrix   : 0.9746
Kygo            vs Drake           : 0.9896
Queen           vs Don Omar        : 0.9756


Interpretation of the three pairs:

The observed cosine similarity values were 0.9746 between Kygo and
Martin Garrix, 0.9896 between Kygo and Drake, and 0.9756 between Queen
and Don Omar. This ordering does not match the initial hypothesis that
same-genre artists would produce the highest similarity. Instead, the
cross-genre pair (Kygo and Drake) scored highest.

Two observations follow from this result.

First, all three similarity values fall within a narrow band between
0.97 and 0.99. This clustering is a direct consequence of using
min-max normalized data. After scaling to [0, 1], every artist
fingerprint sits in the positive orthant of the 9-dimensional feature
space, meaning all vectors point in broadly the same general direction.
Cosine similarity, which measures the angle between vectors, becomes
naturally compressed in this setting and loses some discriminative
power. Z-score normalization (centering to mean 0) would produce a
wider dynamic range, but that is outside the scope of this task.
The relative ordering of the three pairs still carries information,
even if the absolute values are close.

Second, the counter-intuitive ordering reveals something meaningful
about the interaction between genre labels and audio profiles. Kygo
produces tropical house, a chill, moderate-energy, melodic subgenre
of EDM. Martin Garrix produces big-room house, characterized by
extreme energy and aggressive drops. Despite sharing the EDM genre
label, their average audio fingerprints differ noticeably on energy
and valence. Drake, meanwhile, has a large catalog of melodic,
mid-tempo tracks that share more with Kygo's smooth production style
than Kygo shares with Martin Garrix's high-intensity output. Queen
and Don Omar, though on paper very distant (rock versus latin), both
produce studio-recorded music with prominent vocals and moderate
energy, which explains their moderately high similarity.

The practical takeaway: genre labels are a useful first-pass filter
for recommendation systems, but they are not sufficient on their own.
Two artists in the same genre can have substantially different sonic
profiles, while two artists from different genres can converge on
similar audio characteristics. A robust recommendation engine should
combine genre metadata with audio-feature similarity rather than
relying on either signal alone.

# Bonus 2 — Genre Cluster Profile

In this section, I move from artist-level fingerprints to genre-level
centroids. Each genre is represented by the mean vector of its nine
audio features, and pairwise Euclidean distances between all genres are
computed via NumPy broadcasting, with no explicit loops. This gives a
compact geometric picture of how genres relate to each other in
audio-feature space.

The same normalized data used in Bonus 1 is reused here for the same
reason: Euclidean distance is not scale-invariant, and features on
different scales would otherwise distort the geometry.

In [55]:
# Compute mean of each audio feature per genre. Result is a
# (n_genres, 9) matrix where each row is one genre's centroid.
genre_centroids_df = df_norm.groupby('playlist_genre')[audio_features].mean()
genre_labels = genre_centroids_df.index.tolist()
centroids = genre_centroids_df.to_numpy()

print(f"Centroid matrix shape: {centroids.shape}")
print(f"Genres: {genre_labels}\n")
print("Genre Centroid Matrix (normalized features):")
print("=" * 70)
print(genre_centroids_df.round(4))

Centroid matrix shape: (6, 9)
Genres: ['edm', 'latin', 'pop', 'r&b', 'rap', 'rock']

Genre Centroid Matrix (normalized features):
                danceability  energy  loudness  speechiness  acousticness  \
playlist_genre                                                              
edm                   0.6664  0.8024    0.8596       0.0944        0.0820   
latin                 0.7256  0.7083    0.8420       0.1118        0.2122   
pop                   0.6504  0.7010    0.8410       0.0806        0.1718   
r&b                   0.6818  0.5909    0.8085       0.1272        0.2615   
rap                   0.7308  0.6506    0.8257       0.2151        0.1936   
rock                  0.5296  0.7328    0.8143       0.0628        0.1461   

                instrumentalness  liveness  valence   tempo  
playlist_genre                                               
edm                       0.2199    0.2127   0.4043  0.5253  
latin                     0.0447    0.1814   0.6110  0.4954  
pop  

In [56]:
# Fully vectorized pairwise Euclidean distance, no loops.
#
# Broadcasting trick:
#   centroids[:, np.newaxis, :]  has shape (n, 1, 9)
#   centroids[np.newaxis, :, :]  has shape (1, n, 9)
#   subtraction broadcasts to    shape (n, n, 9) of pairwise differences
# Then square, sum along the feature axis, and take sqrt.
diff = centroids[:, np.newaxis, :] - centroids[np.newaxis, :, :]
distance_matrix = np.sqrt(np.sum(diff ** 2, axis=-1))

distance_df = pd.DataFrame(
    distance_matrix, index=genre_labels, columns=genre_labels
).round(4)

print(f"Distance matrix shape: {distance_matrix.shape}")
print("\nPairwise Euclidean Distance Between Genres:")
print("=" * 70)
print(distance_df)

Distance matrix shape: (6, 6)

Pairwise Euclidean Distance Between Genres:
          edm   latin     pop     r&b     rap    rock
edm    0.0000  0.3244  0.2390  0.3719  0.2965  0.2730
latin  0.3244  0.0000  0.1387  0.1604  0.1616  0.2299
pop    0.2390  0.1387  0.0000  0.1639  0.1682  0.1396
r&b    0.3719  0.1604  0.1639  0.0000  0.1501  0.2545
rap    0.2965  0.1616  0.1682  0.1501  0.0000  0.2730
rock   0.2730  0.2299  0.1396  0.2545  0.2730  0.0000


In [57]:
# Mask the diagonal (self-distance = 0) and use only the upper triangle
# to avoid counting each pair twice, since the distance matrix is symmetric.
mask = np.triu(np.ones_like(distance_matrix, dtype=bool), k=1)
distance_masked = np.where(mask, distance_matrix, np.nan)

min_idx = np.unravel_index(np.nanargmin(distance_masked), distance_masked.shape)
max_idx = np.unravel_index(np.nanargmax(distance_masked), distance_masked.shape)

closest_pair  = (genre_labels[min_idx[0]], genre_labels[min_idx[1]])
farthest_pair = (genre_labels[max_idx[0]], genre_labels[max_idx[1]])

print(f"Most similar genres  : {closest_pair[0]} and {closest_pair[1]} "
      f"(distance = {distance_matrix[min_idx]:.4f})")
print(f"Most different genres: {farthest_pair[0]} and {farthest_pair[1]} "
      f"(distance = {distance_matrix[max_idx]:.4f})")

Most similar genres  : latin and pop (distance = 0.1387)
Most different genres: edm and r&b (distance = 0.3719)


In [58]:
def recommend_similar_genre(genre, top_k=3):
    """
    Recommend the top-k genres most similar to a given genre.

    Uses the pairwise Euclidean distance matrix computed from normalized
    genre centroids to rank all other genres by audio profile similarity.
    Smaller distances indicate more similar sonic characteristics.

    Parameters
    ----------
    genre : str
        The reference genre. Must match one of the labels in the
        module-level ``genre_labels`` list.
    top_k : int, optional
        Number of similar genres to return. Default is 3. Must be less
        than the total number of available genres.

    Returns
    -------
    list of tuple
        A list of ``(genre_name, distance)`` tuples, sorted from most
        similar (smallest distance) to least similar. The reference
        genre itself is excluded from the result.

    Raises
    ------
    ValueError
        If ``genre`` is not found in the ``genre_labels`` list.

    Examples
    --------
    >>> recommend_similar_genre("pop", top_k=2)
    [('latin', 0.1234), ('r&b', 0.2345)]

    Notes
    -----
    This function depends on the module-level variables
    ``distance_matrix`` and ``genre_labels``, which must be computed
    beforehand from the genre centroid matrix.
    """
    if genre not in genre_labels:
        raise ValueError(
            f"Genre '{genre}' not found. Available genres: {genre_labels}"
        )

    idx = genre_labels.index(genre)
    distances = distance_matrix[idx]

    ranked = [
        (genre_labels[i], distances[i])
        for i in range(len(genre_labels)) if i != idx
    ]
    ranked.sort(key=lambda x: x[1])
    return ranked[:top_k]


# Demonstrate the function on every genre
print("Genre Recommendations Based on Audio Profile Similarity:")
print("=" * 70)
for g in genre_labels:
    recs = recommend_similar_genre(g, top_k=3)
    rec_str = ", ".join([f"{name} ({dist:.4f})" for name, dist in recs])
    print(f"{g:<8} -> {rec_str}")

Genre Recommendations Based on Audio Profile Similarity:
edm      -> pop (0.2390), rock (0.2730), rap (0.2965)
latin    -> pop (0.1387), r&b (0.1604), rap (0.1616)
pop      -> latin (0.1387), rock (0.1396), r&b (0.1639)
r&b      -> rap (0.1501), latin (0.1604), pop (0.1639)
rap      -> r&b (0.1501), latin (0.1616), pop (0.1682)
rock     -> pop (0.1396), latin (0.2299), r&b (0.2545)


Interpretation of genre proximity:

The most similar genres are latin and pop, with a Euclidean distance of
0.1387. The most different pair is edm and r&b, at 0.3719 — nearly
three times further apart. Both results align with the musical
intuition built up during Stage 2. Latin and pop share high popularity,
high valence, and comparable danceability, so their centroids sit close
together. Edm and r&b occupy opposite corners of the sound spectrum:
edm has the highest energy and lowest valence among all genres, while
r&b sits at moderate energy with warmer valence and richer acoustic
character.

Three broader patterns emerge from the distance matrix.

First, pop functions as a universal middle-ground genre. It appears as
the top-1 nearest neighbor for edm, latin, and rock, and shows up in
the top-3 for every other genre. This matches its role in the music
industry: pop is designed to absorb and blend elements from every other
genre, so its centroid lands near the center of the audio-feature space.

Second, edm is an isolated cluster. Even its closest neighbor (pop) is
at distance 0.2390, whereas most other genres can find a neighbor at
distance 0.13 to 0.16. This isolation reflects edm's extreme profile:
consistently high energy, high loudness, and unusually low valence and
danceability variance. From a recommendation standpoint, this means
cross-genre discovery from edm is inherently harder — there is no
naturally close neighbor to bridge into.

Third, rap and r&b are the closest cross-genre pair after latin and pop,
at distance 0.1501. This confirms the hip-hop family relationship
between the two, driven by shared speechiness patterns, moderate energy,
and similar production styles. Recommendation systems can rely on this
proximity when moving listeners between rap and r&b playlists.

The recommend_similar_genre function turns these insights into a
reusable utility: given any genre, it returns the top-k sonically
closest neighbors ranked by distance. This can serve as the backbone
for cross-genre discovery features, playlist bridge tracks, or
onboarding flows that gradually widen a new listener's exposure.

# Bonus 3 — Reusable Analysis Class

This section refactors the entire analysis (Stages 1 to 3 plus Bonus 2)
into a single Python class called SpotifyAnalyzer. The class loads and
preprocesses the data on instantiation, exposes one method per analysis
stage, and offers a generate_report method that consolidates every key
insight into a single dictionary ready to be serialized to JSON. All
methods use type hints, and normalization is applied lazily so that
downstream methods only pay the computation cost once.

In [ ]:
import pandas as pd
import numpy as np
import json
from typing import Optional


class SpotifyAnalyzer:
    """
    Exploratory data analysis toolkit for a Spotify tracks dataset.

    Loads a CSV of Spotify tracks from a URL, cleans the data
    (dropping high-missingness columns and duplicate rows), and exposes
    methods for summary statistics, genre analysis, audio-feature
    correlation, inter-genre similarity via Euclidean distance on
    min-max-normalised audio features, and full JSON-safe report
    generation.

    Parameters
    ----------
    url : str
        URL pointing to a CSV file of Spotify tracks.

    Attributes
    ----------
    AUDIO_FEATURES : list of str
        Names of the nine audio-feature columns used for normalisation,
        correlation, and genre-similarity analyses.
    url : str
        The source URL passed at construction time.
    df : pandas.DataFrame
        The cleaned Spotify tracks DataFrame (high-missingness columns
        and duplicates removed).

    Examples
    --------
    >>> analyzer = SpotifyAnalyzer("https://example.com/spotify.csv")
    >>> report = analyzer.generate_report()
    """

    AUDIO_FEATURES = [
        'danceability', 'energy', 'loudness', 'speechiness',
        'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
    ]

    def __init__(self, url: str) -> None:
        """
        Load the dataset from a URL and run initial preprocessing.

        Reads the CSV at ``url`` into a DataFrame, initialises internal
        caches for the normalised feature matrix, genre labels, and
        distance matrix to ``None``, and immediately calls
        ``_preprocess`` to clean the data.

        Parameters
        ----------
        url : str
            URL pointing to a CSV file of Spotify tracks.
        """
        self.url: str = url
        self.df: pd.DataFrame = pd.read_csv(url)
        self._X_norm: Optional[np.ndarray] = None
        self._genre_labels: Optional[list] = None
        self._distance_matrix: Optional[np.ndarray] = None
        self._preprocess()

    def _preprocess(self) -> None:
        """
        Clean the loaded DataFrame in place.

        Performs two cleaning steps:
        1. Drops every column whose percentage of missing values exceeds
           20%.
        2. Removes duplicate rows and resets the index.

        This method is called automatically by ``__init__`` and should
        not normally be called directly.
        """
        missing_pct = (self.df.isnull().sum() / len(self.df)) * 100
        cols_to_drop = missing_pct[missing_pct > 20.0].index.tolist()
        if cols_to_drop:
            self.df = self.df.drop(columns=cols_to_drop)
        self.df = self.df.drop_duplicates().reset_index(drop=True)

    def _ensure_normalized(self) -> None:
        """
        Lazily compute and cache the min-max-normalised audio-feature matrix.

        On the first call, extracts the columns listed in
        ``AUDIO_FEATURES`` as a NumPy array and applies column-wise
        min-max normalisation (mapping each feature to the 0-1 range).
        The result is stored in ``_X_norm``. Subsequent calls are
        no-ops.
        """
        if self._X_norm is None:
            X = self.df[self.AUDIO_FEATURES].to_numpy()
            self._X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

    def compute_summary_statistics(self) -> pd.DataFrame:
        """
        Compute descriptive statistics with IQR for all numeric columns.

        Selects every numeric column in the DataFrame and calculates
        the mean, median, standard deviation, and interquartile range
        (IQR). IQR is computed manually as the difference between the
        75th and 25th percentiles via ``np.percentile``, with NaN values
        dropped before calculation.

        Returns
        -------
        pandas.DataFrame
            A DataFrame indexed by column name with columns ``mean``,
            ``median``, ``std``, and ``IQR``. All values are rounded
            to four decimal places.
        """
        numeric_df = self.df.select_dtypes(include=np.number)
        summary = pd.DataFrame({
            'mean':   numeric_df.mean(),
            'median': numeric_df.median(),
            'std':    numeric_df.std(),
            'IQR':    numeric_df.apply(
                lambda col: np.percentile(col.dropna(), 75) - np.percentile(col.dropna(), 25)
            )
        })
        return summary.round(4)

    def analyze_genres(self) -> dict:
        """
        Analyse genre-level statistics, top artists, and hit-potential tracks.

        Performs three analyses:
        1. Genre statistics - groups tracks by ``playlist_genre`` and
           computes mean, std, and median for ``track_popularity``,
           ``danceability``, ``energy``, and ``valence``. Identifies
           the genre with the highest popularity standard deviation.
        2. Top-10 artists - selects the 10 most frequent artists by
           track count, computes their mean popularity, and calculates
           the Pearson correlation between track count and mean
           popularity (volume-vs-quality).
        3. Hit-potential filter - isolates tracks with popularity
           greater than 70, danceability greater than 0.7, energy
           greater than 0.6, and duration less than 240000 ms, then
           reports the count and per-genre distribution.

        Returns
        -------
        dict
            Keys and value types:
            - ``'genre_stats'`` : pandas.DataFrame with MultiIndex
              columns ``(feature, agg)``.
            - ``'highest_popularity_variance_genre'`` : str
            - ``'highest_popularity_variance_value'`` : float
            - ``'top_10_artists'`` : pandas.DataFrame with columns
              ``track_count`` and ``mean_popularity``, sorted by
              descending mean popularity.
            - ``'volume_vs_quality_correlation'`` : float
            - ``'hit_potential_count'`` : int
            - ``'hit_potential_dominant_genre'`` : str
            - ``'hit_potential_genre_distribution'`` : dict mapping
              genre names to counts.
        """
        features = ['track_popularity', 'danceability', 'energy', 'valence']
        genre_stats = (
            self.df.groupby('playlist_genre')[features]
            .agg(['mean', 'std', 'median']).round(4)
        )
        pop_variance = genre_stats[('track_popularity', 'std')]
        top_var_genre = pop_variance.idxmax()

        top_10_artists = self.df['track_artist'].value_counts().head(10)
        artist_pop = (
            self.df[self.df['track_artist'].isin(top_10_artists.index)]
            .groupby('track_artist')['track_popularity'].mean().round(2)
        )
        top_10_summary = pd.DataFrame({
            'track_count': top_10_artists,
            'mean_popularity': artist_pop
        }).sort_values('mean_popularity', ascending=False)
        volume_quality_corr = float(
            top_10_summary['track_count'].corr(top_10_summary['mean_popularity'])
        )

        hit_mask = (
            (self.df['track_popularity'] > 70) &
            (self.df['danceability'] > 0.7) &
            (self.df['energy'] > 0.6) &
            (self.df['duration_ms'] < 240000)
        )
        hit_tracks = self.df[hit_mask]
        genre_dist = hit_tracks['playlist_genre'].value_counts()

        return {
            'genre_stats': genre_stats,
            'highest_popularity_variance_genre': top_var_genre,
            'highest_popularity_variance_value': float(pop_variance.max()),
            'top_10_artists': top_10_summary,
            'volume_vs_quality_correlation': volume_quality_corr,
            'hit_potential_count': int(len(hit_tracks)),
            'hit_potential_dominant_genre': genre_dist.idxmax(),
            'hit_potential_genre_distribution': genre_dist.to_dict()
        }

    def analyze_audio_features(self) -> dict:
        """
        Analyse audio-feature correlations and high-energy popularity.

        Performs two analyses on the nine audio features:
        1. Correlation matrix - computes the Pearson correlation
           between all nine audio features using ``np.corrcoef`` on the
           original (un-normalised) values, and identifies the strongest
           positive and strongest negative pair using an upper-triangle
           mask to avoid double-counting symmetric pairs.
        2. High-energy subset - uses NumPy boolean masking to isolate
           tracks with energy above the mean plus one standard
           deviation, then compares the mean ``track_popularity`` of
           this subset against the overall mean.

        Side effect: ensures ``_X_norm`` is populated for consistency
        with downstream methods, even though the analyses in this
        method operate on the original feature values.

        Returns
        -------
        dict
            Keys and value types:
            - ``'correlation_matrix'`` : pandas.DataFrame, 9 x 9,
              rounded to four decimal places.
            - ``'strongest_positive_pair'`` : tuple of (str, str, float).
            - ``'strongest_negative_pair'`` : tuple of (str, str, float).
            - ``'high_energy_threshold'`` : float
            - ``'high_energy_track_count'`` : int
            - ``'high_energy_mean_popularity'`` : float
            - ``'overall_mean_popularity'`` : float
            - ``'popularity_difference'`` : float
        """
        self._ensure_normalized()

        X = self.df[self.AUDIO_FEATURES].to_numpy()
        corr_matrix = np.corrcoef(X, rowvar=False)
        corr_df = pd.DataFrame(
            corr_matrix, index=self.AUDIO_FEATURES, columns=self.AUDIO_FEATURES
        ).round(4)

        mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
        corr_masked = np.where(mask, corr_matrix, np.nan)
        max_idx = np.unravel_index(np.nanargmax(corr_masked), corr_masked.shape)
        min_idx = np.unravel_index(np.nanargmin(corr_masked), corr_masked.shape)

        energy_arr = self.df['energy'].to_numpy()
        popularity_arr = self.df['track_popularity'].to_numpy()
        threshold = float(energy_arr.mean() + energy_arr.std())
        high_mask = energy_arr > threshold
        high_pop_mean = float(popularity_arr[high_mask].mean())
        overall_pop_mean = float(popularity_arr.mean())

        return {
            'correlation_matrix': corr_df,
            'strongest_positive_pair': (
                self.AUDIO_FEATURES[max_idx[0]],
                self.AUDIO_FEATURES[max_idx[1]],
                float(corr_matrix[max_idx])
            ),
            'strongest_negative_pair': (
                self.AUDIO_FEATURES[min_idx[0]],
                self.AUDIO_FEATURES[min_idx[1]],
                float(corr_matrix[min_idx])
            ),
            'high_energy_threshold': threshold,
            'high_energy_track_count': int(high_mask.sum()),
            'high_energy_mean_popularity': high_pop_mean,
            'overall_mean_popularity': overall_pop_mean,
            'popularity_difference': high_pop_mean - overall_pop_mean
        }

    def analyze_genre_similarity(self) -> dict:
        """
        Compute pairwise genre similarity using Euclidean distance on normalised features.

        Min-max-normalises the nine ``AUDIO_FEATURES``, computes
        per-genre centroids, and builds a pairwise Euclidean distance
        matrix via broadcasting. The most similar and most different
        genre pairs are extracted from the upper triangle of the matrix.

        Side effects: caches ``_X_norm``, ``_genre_labels``, and
        ``_distance_matrix`` for reuse by ``recommend_similar_genre``.

        Returns
        -------
        dict
            Keys and value types:
            - ``'genre_centroids'`` : pandas.DataFrame indexed by genre
              with one column per audio feature, rounded to four
              decimal places.
            - ``'distance_matrix'`` : pandas.DataFrame of pairwise
              Euclidean distances, rounded to four decimal places.
            - ``'genre_labels'`` : list of str
            - ``'most_similar_pair'`` : tuple of (str, str, float).
            - ``'most_different_pair'`` : tuple of (str, str, float).
        """
        self._ensure_normalized()

        df_norm = self.df.copy()
        df_norm[self.AUDIO_FEATURES] = self._X_norm

        centroids_df = df_norm.groupby('playlist_genre')[self.AUDIO_FEATURES].mean().round(4)
        self._genre_labels = centroids_df.index.tolist()
        centroids = centroids_df.to_numpy()

        diff = centroids[:, np.newaxis, :] - centroids[np.newaxis, :, :]
        self._distance_matrix = np.sqrt(np.sum(diff ** 2, axis=-1))

        distance_df = pd.DataFrame(
            self._distance_matrix,
            index=self._genre_labels, columns=self._genre_labels
        ).round(4)

        mask = np.triu(np.ones_like(self._distance_matrix, dtype=bool), k=1)
        distance_masked = np.where(mask, self._distance_matrix, np.nan)
        min_idx = np.unravel_index(np.nanargmin(distance_masked), distance_masked.shape)
        max_idx = np.unravel_index(np.nanargmax(distance_masked), distance_masked.shape)

        return {
            'genre_centroids': centroids_df,
            'distance_matrix': distance_df,
            'genre_labels': self._genre_labels,
            'most_similar_pair': (
                self._genre_labels[min_idx[0]],
                self._genre_labels[min_idx[1]],
                float(self._distance_matrix[min_idx])
            ),
            'most_different_pair': (
                self._genre_labels[max_idx[0]],
                self._genre_labels[max_idx[1]],
                float(self._distance_matrix[max_idx])
            )
        }

    def recommend_similar_genre(self, genre: str, top_k: int = 3) -> list:
        """
        Recommend the most sonically similar genres to a given genre.

        Ranks all other genres by ascending Euclidean distance (from
        the cached distance matrix) and returns the closest ``top_k``
        entries. If ``analyze_genre_similarity`` has not yet been
        called, it is invoked automatically to populate the distance
        matrix.

        Parameters
        ----------
        genre : str
            Name of the reference genre. Must match one of the values
            in the ``playlist_genre`` column.
        top_k : int, optional
            Number of nearest genres to return. Default is 3.

        Returns
        -------
        list of tuple
            Each element is a ``(genre_name, distance)`` pair, sorted
            by ascending distance.

        Raises
        ------
        ValueError
            If ``genre`` is not present in the dataset's genre labels.
        """
        if self._distance_matrix is None:
            self.analyze_genre_similarity()

        if genre not in self._genre_labels:
            raise ValueError(
                f"Genre '{genre}' not found. Available: {self._genre_labels}"
            )

        idx = self._genre_labels.index(genre)
        distances = self._distance_matrix[idx]
        ranked = [
            (self._genre_labels[i], float(distances[i]))
            for i in range(len(self._genre_labels)) if i != idx
        ]
        ranked.sort(key=lambda x: x[1])
        return ranked[:top_k]

    def _to_json_safe(self, obj):
        """
        Recursively convert an object to a JSON-serialisable structure.

        Handles nested dicts, lists, tuples, NumPy arrays, pandas
        DataFrames (including MultiIndex columns), Series, and NumPy
        scalar types. DataFrames are converted to
        ``{index: {col: value, ...}, ...}`` dictionaries.
        Unrecognised types are cast to ``str`` as a fallback.

        Parameters
        ----------
        obj : object
            The value to convert. May be arbitrarily nested.

        Returns
        -------
        object
            A structure composed only of ``dict``, ``list``, ``int``,
            ``float``, ``str``, ``bool``, and ``None`` - all natively
            serialisable by ``json.dumps``.
        """
        if isinstance(obj, dict):
            return {str(k): self._to_json_safe(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [self._to_json_safe(x) for x in obj]
        if isinstance(obj, np.ndarray):
            return self._to_json_safe(obj.tolist())
        if isinstance(obj, pd.DataFrame):
            df = obj.copy()
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = ['_'.join(map(str, col)).strip('_') for col in df.columns]
            return {
                str(idx): self._to_json_safe(row.to_dict())
                for idx, row in df.iterrows()
            }
        if isinstance(obj, pd.Series):
            return {str(k): self._to_json_safe(v) for k, v in obj.to_dict().items()}
        if isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        if isinstance(obj, (int, float, str, bool)) or obj is None:
            return obj
        return str(obj)

    def generate_report(self) -> dict:
        """
        Run the full analysis pipeline and return a JSON-safe report.

        Aggregates results from ``compute_summary_statistics``,
        ``analyze_genres``, ``analyze_audio_features``,
        ``analyze_genre_similarity``, and ``recommend_similar_genre``
        (for every genre with ``top_k=3``), adds dataset metadata
        (source URL, shape, and column list), and converts the entire
        structure to JSON-serialisable Python primitives via
        ``_to_json_safe``.

        Returns
        -------
        dict
            Top-level keys:
            - ``'dataset_info'`` : dict with ``'source_url'`` (str),
              ``'shape'`` (list of int), and ``'columns'`` (list of
              str).
            - ``'summary_statistics'`` : dict (from DataFrame).
            - ``'genre_analysis'`` : dict.
            - ``'audio_feature_analysis'`` : dict.
            - ``'genre_similarity_analysis'`` : dict.
            - ``'sample_recommendations'`` : dict mapping each genre
              to a list of ``[genre_name, distance]`` pairs.
        """
        report = {
            'dataset_info': {
                'source_url': self.url,
                'shape': list(self.df.shape),
                'columns': list(self.df.columns)
            },
            'summary_statistics': self.compute_summary_statistics(),
            'genre_analysis': self.analyze_genres(),
            'audio_feature_analysis': self.analyze_audio_features(),
            'genre_similarity_analysis': self.analyze_genre_similarity(),
            'sample_recommendations': {
                genre: self.recommend_similar_genre(genre, top_k=3)
                for genre in self._genre_labels
            }
        }
        return self._to_json_safe(report)


# Demo: instantiate, generate report, verify JSON serialization
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"
analyzer = SpotifyAnalyzer(url)
report = analyzer.generate_report()

print("Report top-level keys:")
for k, v in report.items():
    if isinstance(v, dict):
        print(f"  {k}: dict with {len(v)} keys")
    elif isinstance(v, list):
        print(f"  {k}: list of {len(v)} items")
    else:
        print(f"  {k}: {type(v).__name__}")

json_str = json.dumps(report, indent=2)
print(f"\nReport serialized to JSON: {len(json_str):,} characters")

print("\nSample of JSON output (first 2000 chars):")
print(json_str[:2000])

## Docstring Generation for SpotifyAnalyzer via Antigravity

Tool used: Antigravity (Google's agentic IDE, successor to Gemini Code
Assist, powered by Gemini models). One prompt was used to generate
docstrings for the entire class at once, in order to evaluate both
per-method quality and overall consistency across methods.

### Prompt sent to Antigravity

> In my notebook, there is a class called SpotifyAnalyzer used for
> performing exploratory data analysis on a Spotify tracks dataset.
> Generate comprehensive NumPy-style docstrings for the class itself
> and for every method, including __init__ and the private helper
> methods. Make sure each docstring accurately reflects the method's
> actual behavior, parameters, and return type based on the code.
>
> Just show me the docstrings as code blocks, don't edit the file yet.

### Evaluation of the raw AI output

The output was strong on most methods but revealed one significant gap
and several stylistic issues that required manual correction.

The most important issue was an omission: the AI generated docstrings
for the class itself, ``__init__``, ``_preprocess``,
``_ensure_normalized``, ``compute_summary_statistics``,
``analyze_genres``, ``analyze_genre_similarity``,
``recommend_similar_genre``, ``_to_json_safe``, and ``generate_report``,
but completely skipped ``analyze_audio_features``. This is one of the
five main analysis methods and is explicitly referenced in
``generate_report``. The omission was silent (no warning, no note) and
would have been easy to miss on a quick scan. I wrote the docstring
for ``analyze_audio_features`` from scratch, mirroring the AI's style
in the other methods to keep the class consistent.

The second issue was stylistic. The docstring for ``analyze_genres``
used bold markdown syntax (``**Genre statistics**``, ``**Top-10
artists**``, ``**Hit-potential filter**``) to label the three
sub-analyses. Bold markdown does not render inside a docstring when
accessed via ``help()`` or ``?``, and produces visual noise in plain
text. I replaced the bold syntax with plain capitalised labels
followed by a hyphen.

The third issue was conformance to NumPy docstring style. Every
docstring ran the summary line directly into the extended description
without the blank line separator that NumPy style requires. I added
blank lines throughout to bring the docstrings into strict conformance.

Two minor cleanups were applied for consistency with the rest of the
codebase, which uses ASCII characters exclusively. The AI used em
dashes (``–``) in a few places, which I replaced with hyphens (``-``),
and it wrote ``20 %`` with a space, which I normalised to ``20%``.

### Overall assessment

The AI was strongest when documenting methods with clearly structured
return values (``analyze_genres``, ``analyze_genre_similarity``,
``generate_report``), where it listed every dictionary key and its
type accurately. It correctly identified side effects (caching
behavior in ``_ensure_normalized`` and ``analyze_genre_similarity``)
and correctly noted the auto-invocation pattern in
``recommend_similar_genre``. The Raises section on
``recommend_similar_genre`` was a nice touch that I would not
necessarily have thought to add manually.

The weaknesses were the silent omission of an entire method's docstring
and the stylistic mismatches (bold markdown, non-ASCII characters,
missing blank lines). Both categories underline a broader point about
AI-assisted documentation: the output looks polished and comprehensive
at first glance, but a method-by-method audit against the actual code
is necessary. In this case, running the audit caught a missing docstring
that would otherwise have shipped with the notebook.

### Bonus 1, 2, and 3 Summary

The three bonus sections together extend the analysis from population
statistics into vector-space representations, geometric relationships
between genres, and a fully reusable engineering artifact.

In Bonus 1, each artist was reduced to a 9-dimensional fingerprint
vector, and cosine similarity was implemented directly from its
mathematical definition. The three tested pairs (Kygo vs Martin Garrix,
Kygo vs Drake, Queen vs Don Omar) produced values in a narrow band
(0.9746 to 0.9896), a consequence of using min-max normalized data that
places all vectors in the positive orthant. The relative ordering
revealed that same-genre labels do not guarantee highest similarity:
Kygo's tropical house profile turned out closer to Drake's melodic rap
than to Martin Garrix's big-room house. This finding supports a
recommendation strategy that combines genre metadata with audio-feature
similarity rather than relying on genre labels alone.

In Bonus 2, each genre was represented by its centroid, and pairwise
Euclidean distances were computed via NumPy broadcasting with no loops.
Latin and pop emerged as the most similar pair (0.1387), while edm and
r&b were the most different (0.3719). Three structural patterns became
visible: pop acts as a universal middle-ground genre, edm forms an
isolated cluster with no naturally close neighbor, and rap and r&b sit
close together as expected of the hip-hop family. The
recommend_similar_genre function operationalizes the distance matrix
into a reusable recommendation utility.

In Bonus 3, the entire analysis pipeline was refactored into the
SpotifyAnalyzer class. On instantiation it loads the dataset, drops
columns with more than 20 percent missing values, and removes exact
duplicate rows. Each analysis stage is exposed as a public method with
full type hints. Normalization is applied lazily so that downstream
methods only pay the computation cost once. The generate_report method
consolidates every key insight into a single JSON-safe dictionary via
a recursive _to_json_safe helper that handles NumPy scalars, arrays,
Pandas Series, and DataFrames with MultiIndex columns. The class runs
end-to-end from a single Colab cell with no additional setup.

Taken together with all prior stages, the notebook forms a complete
EDA pipeline from raw CSV to structured insights. The three findings
highlighted for a non-technical audience in Section 4.2 all fall
directly out of this pipeline and can be reproduced end-to-end by
running SpotifyAnalyzer.generate_report from a single cell.